# 01 - Benchmark de Avaliação Limpa, Robustez a Ruído e Profiling de Inferência

Reproduz as avaliações de fragmentos curtos (512 bp) que sustentam as **Tabelas 1, 2, 3 e 4** e a **Figura 1** do manuscrito.
- Avalia os 5 modelos de espécie e os 2 modelos de gênero em $\epsilon \in \{0.0, 0.02, 0.05, 0.10, 0.15\}$.
- Perfila eficiência computacional (parâmetros, latência, throughput, GFLOPs).
- Particiona as faixas de F1 da Tabela 4 (, , , ).
- Executa comparação pareada especialista entre ResNet50 e ConvNeXt resolvendo vitórias e empates.


In [ ]:
import os
import sys
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import torch
import pandas as pd
import numpy as np
import json
from datetime import datetime, timezone

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
    resolve_split_files,
    create_optimized_dataloader,
    free_gpu_memory,
)
from viral_cnn.models.registry import (
    REGISTERED_VARIANTS,
    build_registered_model,
)
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.training.profiler import profile_inference_model
from viral_cnn.eval.runner import evaluate_dataloader
from viral_cnn.eval.uncertainty import bootstrap_classification_intervals
from viral_cnn.eval.runner import save_c04_query_evidence
from viral_cnn.eval.metrics import (
    compute_species_f1_bands,
    compute_species_specialist_comparison,
)

# 1. Initialize Runtime and Environment (AD-017)
# Por padrão (REMOTE_SERVER != False), conecta automaticamente à topologia do servidor remoto (/home/mlima/workspace)
# e desativa o modo smoke test (SMOKE_TEST=0).
# Para rodar em modo local de teste sintético rápido em CPU, defina: os.environ['REMOTE_SERVER'] = 'False'
from viral_cnn.m02_workflow import begin_m02
run = begin_m02("short/01")
ctx = run.ctx
output_dir = run.output_dir
c04_run_id = run.run_id
print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")


In [ ]:
with run.guard():
    # 2. Setup Artifacts and Data Loaders
    if ctx.is_smoke_test:
        run.prepare_smoke()



    species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
    genus_parquet_dir = Path(ctx.topology.paths["short"]["genus_parquet_dir"])

    species_test_files = run.split_files("species", "test")
    genus_test_files = run.split_files("genus", "test")

    test_ds_species = ParquetBatchIterableDataset(
        species_test_files,
        batch_size=32 if ctx.is_smoke_test else 4096,
        shuffle=False,
        split="test",
        include_identity=True,
    )
    test_loader_species = create_optimized_dataloader(
        test_ds_species,
        is_smoke_test=ctx.is_smoke_test,
        device=ctx.device,
    )

    test_ds_genus = ParquetBatchIterableDataset(
        genus_test_files,
        batch_size=32 if ctx.is_smoke_test else 4096,
        shuffle=False,
        split="test",
        include_identity=True,
    )
    test_loader_genus = create_optimized_dataloader(
        test_ds_genus,
        is_smoke_test=ctx.is_smoke_test,
        device=ctx.device,
    )
    print("DataLoaders successfully configured.")


In [ ]:
with run.guard():
    # 3. Model Profiling (Table 2)
    profiling_results = []
    species_variants = [
        "resnet_clean_species",
        "resnet_noise_species",
        "convnext_zero_species",
        "convnext_transfer_species",
        "convnext_noise_species",
    ]
    genus_variants = [
        "resnet_genus",
        "convnext_genus",
    ]

    if not ctx.is_smoke_test:
        for name in species_variants + genus_variants:
            ckpt_cand = ctx.topology.paths["checkpoints"].get(name)
            if not ckpt_cand or not Path(ckpt_cand).exists():
                raise FileNotFoundError(f"Required checkpoint for '{name}' not found at: {ckpt_cand}")

    for name in species_variants + genus_variants:
        ckpt = ctx.topology.paths["checkpoints"].get(name) if not ctx.is_smoke_test else None
        model = build_registered_model(name, checkpoint_path=ckpt, device=ctx.device)
        prof = profile_inference_model(
            model,
            input_size=(32, 512),
            device=ctx.device,
            num_warmup=2 if ctx.is_smoke_test else 5,
            num_runs=5 if ctx.is_smoke_test else 20,
        )
        prof["model_name"] = name
        profiling_results.append(prof)
        print(f"Profiled {name}: {prof['total_params']:,} params, {prof['avg_latency_ms']:.2f} ms")
        free_gpu_memory(model)
        del model

    df_prof = pd.DataFrame(profiling_results)
    df_prof.to_csv(output_dir / "table2_profiling.csv", index=False)
    print("Profiling table saved.")
    print(df_prof.to_string())


In [ ]:
with run.guard():
    # 4. Clean Evaluation and Controlled Noise Robustness (Table 1, Table 3, Fig 1)
    noise_levels = [0.0, 0.02, 0.05, 0.10, 0.15]
    max_batches = 2 if ctx.is_smoke_test else None

    all_evaluations = {}
    summary_rows = []
    c04_evidence_records = []
    c04_evidence_by_cell = {}

    for name in species_variants:
        print(f"Evaluating Species Model: {name}")
        ckpt = ctx.topology.paths["checkpoints"].get(name) if not ctx.is_smoke_test else None
        model = build_registered_model(name, checkpoint_path=ckpt, device=ctx.device)
        num_classes = 5 if ctx.is_smoke_test else 13683
        evals = evaluate_dataloader(
            model,
            test_loader_species,
            device=ctx.device,
            noise_rates=noise_levels,
            seed=42,
            model_name=name,
            num_classes=num_classes,
            max_batches=max_batches,
            store_predictions=ctx.is_smoke_test,
        )
        free_gpu_memory(model)
        del model
        keep_specialist_arrays = name in ("resnet_noise_species", "convnext_noise_species")
        for eps, res in evals.items():
            evidence_key = (name, eps)
            if res.query_ids_arr is not None and not any(str(q).startswith("missing_query_id_") for q in res.query_ids_arr):
                evidence = save_c04_query_evidence(
                    res, output_dir / "query_evidence" / f"{name}_noise_{eps}.parquet",
                    checkpoint=str(ckpt or "synthetic_model_initialization"), split="test", rank="species", num_classes=num_classes,
                )
                c04_evidence_records.append(evidence)
                c04_evidence_by_cell[evidence_key] = evidence
            summary_rows.append({
                "model_name": name,
                "rank": "species",
                "nominal_noise": eps,
                "effective_noise": res.effective_noise_rate,
                "top1_accuracy": res.metrics.top1_accuracy,
                "top5_accuracy": res.metrics.top5_accuracy,
                "macro_f1": res.metrics.macro_f1,
                "weighted_f1": res.metrics.weighted_f1,
                "loss": res.metrics.cross_entropy_loss,
                "n_samples": res.metrics.n_samples,
                "uncertainty": bootstrap_classification_intervals(
                    res.y_true_arr, res.y_pred_arr, num_classes, top5_hit=res.top5_hit_arr,
                    repetitions=2000, seed=42, confidence=0.95,
                ) if res.query_ids_arr is not None and not any(str(q).startswith("missing_query_id_") for q in res.query_ids_arr) else {"status": "unavailable", "reason": "query identity missing"},
            })
            if not ctx.is_smoke_test and res.metrics.n_samples != 1706270:
                raise RuntimeError(f"Species sample count mismatch for {name}: expected 1706270, got {res.metrics.n_samples}")
            if not ctx.is_smoke_test and not (keep_specialist_arrays and eps in (0.05, 0.10)):
                res.y_true_arr = None
                res.y_pred_arr = None
                res.confs_arr = None
                res.top5_hit_arr = None
                res.query_ids_arr = None
                res.predictions = []
        if not ctx.is_smoke_test:
            for res in evals.values():
                res.top5_hit_arr = None
                res.query_ids_arr = None
        all_evaluations[name] = evals

    for name in genus_variants:
        print(f"Evaluating Genus Model: {name}")
        ckpt = ctx.topology.paths["checkpoints"].get(name) if not ctx.is_smoke_test else None
        model = build_registered_model(name, checkpoint_path=ckpt, device=ctx.device)
        num_classes = 3 if ctx.is_smoke_test else 2702
        evals = evaluate_dataloader(
            model,
            test_loader_genus,
            device=ctx.device,
            noise_rates=noise_levels,
            seed=42,
            model_name=name,
            num_classes=num_classes,
            max_batches=max_batches,
            store_predictions=ctx.is_smoke_test,
        )
        free_gpu_memory(model)
        del model
        for eps, res in evals.items():
            evidence_key = (name, eps)
            if res.query_ids_arr is not None and not any(str(q).startswith("missing_query_id_") for q in res.query_ids_arr):
                evidence = save_c04_query_evidence(
                    res, output_dir / "query_evidence" / f"{name}_noise_{eps}.parquet",
                    checkpoint=str(ckpt or "synthetic_model_initialization"), split="test", rank="genus", num_classes=num_classes,
                )
                c04_evidence_records.append(evidence)
                c04_evidence_by_cell[evidence_key] = evidence
            summary_rows.append({
                "model_name": name,
                "rank": "genus",
                "nominal_noise": eps,
                "effective_noise": res.effective_noise_rate,
                "top1_accuracy": res.metrics.top1_accuracy,
                "top5_accuracy": res.metrics.top5_accuracy,
                "macro_f1": res.metrics.macro_f1,
                "weighted_f1": res.metrics.weighted_f1,
                "loss": res.metrics.cross_entropy_loss,
                "n_samples": res.metrics.n_samples,
                "uncertainty": bootstrap_classification_intervals(
                    res.y_true_arr, res.y_pred_arr, num_classes, top5_hit=res.top5_hit_arr,
                    repetitions=2000, seed=42, confidence=0.95,
                ) if res.query_ids_arr is not None and not any(str(q).startswith("missing_query_id_") for q in res.query_ids_arr) else {"status": "unavailable", "reason": "query identity missing"},
            })
            if not ctx.is_smoke_test and res.metrics.n_samples != 816756:
                raise RuntimeError(f"Genus sample count mismatch for {name}: expected 816756, got {res.metrics.n_samples}")
            if not ctx.is_smoke_test:
                res.y_true_arr = None
                res.y_pred_arr = None
                res.confs_arr = None
                res.top5_hit_arr = None
                res.query_ids_arr = None
                res.predictions = []
        if not ctx.is_smoke_test:
            for res in evals.values():
                res.top5_hit_arr = None
                res.query_ids_arr = None
        all_evaluations[name] = evals

    # Publish the same full-precision F1 point estimates used by the intervals.
    for row in summary_rows:
        estimates = row.get("uncertainty", {}).get("estimates", {})
        for metric in ("macro_f1", "weighted_f1"):
            if estimates.get(metric, {}).get("estimate") is not None:
                row[metric] = estimates[metric]["estimate"]
    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(output_dir / "benchmark_clean_noise_summary.csv", index=False)
    print("Benchmark summary saved.")
    print(df_summary.to_string())


In [ ]:
with run.guard():
    # 5. Table 4 F1 Performance Bands at epsilon = 0.05
    f1_bands_rows = []
    for name in species_variants:
        res_05 = all_evaluations[name][0.05]
        for b_key, b_info in res_05.f1_bands["bands"].items():
            f1_bands_rows.append({
                "model_name": name,
                "band_key": b_key,
                "interval": b_info["range"],
                "class_count": b_info["count"],
                "percent": b_info["percent"],
                "mean_confidence": b_info.get("mean_confidence", 0.0),
            })

    df_f1_bands = pd.DataFrame(f1_bands_rows)
    df_f1_bands.to_csv(output_dir / "table4_f1_bands.csv", index=False)
    print("Table 4 F1 bands saved.")
    print(df_f1_bands.to_string())


In [ ]:
with run.guard():
    # 6. Specialist Comparison: ResNet50 vs ConvNeXt (Table 4 / Note 45)
    specialist_levels = [(0.05, "specialist_comparison.csv"), (0.10, "specialist_comparison_eps010.csv")]

    for eps_val, out_filename in specialist_levels:
        res_resnet = all_evaluations["resnet_noise_species"][eps_val]
        res_convnext = all_evaluations["convnext_noise_species"][eps_val]

        y_true = res_resnet.y_true_arr if res_resnet.y_true_arr is not None else [p.truth_label for p in res_resnet.predictions]
        preds_res = res_resnet.y_pred_arr if res_resnet.y_pred_arr is not None else [p.predicted_label for p in res_resnet.predictions]
        preds_conv = res_convnext.y_pred_arr if res_convnext.y_pred_arr is not None else [p.predicted_label for p in res_convnext.predictions]
        confs_res = res_resnet.confs_arr if res_resnet.confs_arr is not None else [p.confidence for p in res_resnet.predictions]
        confs_conv = res_convnext.confs_arr if res_convnext.confs_arr is not None else [p.confidence for p in res_convnext.predictions]

        comp_res = compute_species_specialist_comparison(
            y_true=y_true,
            preds_resnet=preds_res,
            preds_convnext=preds_conv,
            confs_resnet=confs_res,
            confs_convnext=confs_conv,
            num_classes=5 if ctx.is_smoke_test else 13683,
        )

        print(f"--- Specialist Comparison at epsilon = {eps_val:.2f} ---")
        print(f"ResNet Superior: {comp_res['resnet_superior']}")
        print(f"ConvNeXt Superior: {comp_res['convnext_superior']}")
        print(f"Ties: {comp_res['ties']}")
        print(f"Both Zero: {comp_res['both_zero']}")
        print(f"Total Evaluated Species: {comp_res['total_evaluated_species']}")

        df_specialist = pd.DataFrame(comp_res["per_species"])
        df_specialist.to_csv(output_dir / out_filename, index=False)
        print(f"Specialist comparison saved to {out_filename}.")
        print(df_specialist.head(10).to_string())


In [ ]:
with run.guard():
    # 7. Write Execution Manifest
    evaluated_checkpoints = {}
    for name in species_variants + genus_variants:
        ckpt = ctx.topology.paths["checkpoints"].get(name)
        evaluated_checkpoints[name] = str(ckpt) if ckpt else "none"

    expected_cells = [(m, r) for m in species_variants + genus_variants for r in noise_levels]
    observed_cells = {(r["model_name"], r["nominal_noise"]) for r in summary_rows if (r["model_name"], r["nominal_noise"]) in c04_evidence_by_cell and r.get("uncertainty", {}).get("estimates", {}).get("accuracy", {}).get("estimate") is not None}
    for row in summary_rows:
        row["query_evidence"] = c04_evidence_by_cell.get((row["model_name"], row["nominal_noise"]))
    summary_cells = {(r["model_name"], r["nominal_noise"]) for r in summary_rows}
    completeness = [{"model": m, "noise": r, "status": "complete" if (m, r) in observed_cells else "unavailable", "reason": None if (m, r) in observed_cells else ("query identity or compact evidence unavailable" if (m, r) in summary_cells else "condition not evaluated")} for m, r in expected_cells]
    (output_dir / "c04_completeness.json").write_text(json.dumps({"expected": len(expected_cells), "complete": len(observed_cells), "evidence_shards": len(c04_evidence_records), "cells": completeness}, indent=2))
    (output_dir / "c04_uncertainty.json").write_text(json.dumps({"protocol": "c04_stratified_fragment_v1", "rows": summary_rows}, indent=2))
    manifest = write_notebook_manifest(
        notebook_id="01_benchmark_clean_and_noise",
        data_root=ctx.data_root,
        is_smoke_test=ctx.is_smoke_test,
        checkpoints=evaluated_checkpoints,
        input_files=[*species_test_files, *genus_test_files],
        metrics_summary={"evaluated_models": len(species_variants) + len(genus_variants), "noise_levels": noise_levels, "c04_uncertainty_protocol": "c04_stratified_fragment_v1", "completeness": {"expected": len(expected_cells), "complete": len(observed_cells), "evidence_shards": len(c04_evidence_records)}},
        run_id=c04_run_id,
        configuration={"uncertainty_repetitions": 2000, "uncertainty_seed": 42, "uncertainty_confidence": 0.95},
        outputs=[output_dir / "c04_completeness.json", output_dir / "c04_uncertainty.json", *[Path(x["evidence"]) for x in c04_evidence_records], *[Path(x["manifest"]) for x in c04_evidence_records], output_dir / "benchmark_clean_noise_summary.csv"],
        output_json_path=output_dir / "c04_notebook_manifest.json",
    )
    print("Manifest successfully written.")

    if len(observed_cells) != len(expected_cells):
        raise RuntimeError("Incomplete C04 benchmark conditions")
    manifest = run.complete(input_files=[*species_test_files, *genus_test_files],
        checkpoints=evaluated_checkpoints if not ctx.is_smoke_test else {})
